<a href="https://colab.research.google.com/github/pavithraprem45/projects-/blob/main/Pass_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install g++

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
g++ is already the newest version (4:11.2.0-1ubuntu1).
g++ set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
%%writefile password_manager.cpp
#include <iostream>
#include <vector>
#include <string>
#include <cstdlib> // For rand() and srand()
#include <ctime>   // For time()
#include <algorithm> // For remove_if
#include <fstream>  // For file I/O

using namespace std;

// Class definition for PasswordManager
class PasswordManager {
private:
    string masterPassword;
    vector<pair<string, string>> credentials; // Stores username and encrypted password

    // Function to generate a simple random password
    string generateRandomPassword(int length) {
        string password = "";
        for (int i = 0; i < length; ++i) {
            password += 'a' + rand() % 26; // Generate lowercase letters only
        }
        return password;
    }

public:
    PasswordManager();
    bool setMasterPassword();
    string encrypt(const string& text, int key); //encrypt a string using caesar cipher
    string decrypt(const string& text, int key); //decrypt a string using caesar cipher
    void addCredentials();
    void getCredentials();
    void deleteCredentials();
    bool saveToFile(const string& filename = "passwords.txt"); //save credentials to a file
    bool loadFromFile(const string& filename = "passwords.txt"); //load credentials to a file
    void addCredentialsWithGen(); // New function to add with generated password
};

// Constructor implementation
PasswordManager::PasswordManager() {
    srand(time(0)); // Seed the random number generator
}

// Sets the master password for the Password Manager
bool PasswordManager::setMasterPassword() {
    cout << "Set your master password:(numbers only)";

    getline(cin >> ws, masterPassword);  // Read the entire line into masterPassword

    if (masterPassword.length() < 4 ) {
        cout << "Password must be at least 4 characters long." << endl; // Fixed output stream
        return false;
    }

    return true;
}

// Caesar Cipher Encryption
string PasswordManager::encrypt(const string& text, int key) {
    string result = "";
    for (char character : text) { // Iterate over each character in the text
        if (isalpha(character)) { // Only encrypt alphabetic characters
            char base = isupper(character) ? 'A' : 'a'; // Determine the base character ('A' or 'a')
            result += static_cast<char>((character - base + key) % 26 + base);
        } else {
            result += character;
        }
    }
    return result;
}

// Caesar Cipher Decryption - lec17
string PasswordManager::decrypt(const string& text, int key) {
    string result = "";
    for (char c : text) {
        if (isalpha(c)) {
            char base = islower(c) ? 'a' : 'A';
            result += static_cast<char>((c - base + (26 - key)) % 26 + base);
        } else {
            result += c;
        }
    }
    return result;
}


// Adds new credentials to the Password Manager
void PasswordManager::addCredentials() {
    string username;
    cout << "Enter username: ";
    cin >> username;

    string password;
    cout << "Enter password: ";
    cin >> password;

    int key = 3; // Using a fixed key for simplicity
    string encryptedPassword = encrypt(password, key);
    credentials.push_back(make_pair(username, encryptedPassword));
    cout << "Encrypted password stored." << endl; // Fixed output stream
}

// New function to add credentials with a generated password
void PasswordManager::addCredentialsWithGen() {
    string username;
    cout << "Enter username: ";
    cin >> username;

    string password = generateRandomPassword(8); // Generate a simple password
    cout << "Generated password: " << password << endl;

    int key = 3;
    string encryptedPassword = encrypt(password, key);
    credentials.push_back(make_pair(username, encryptedPassword));
    cout << "Encrypted password stored." << endl; // Fixed output stream
}

// Retrieves credentials from the Password Manager
void PasswordManager::getCredentials() {
    string username;
    cout << "Enter username to retrieve password: ";
    cin >> username;

    for (const auto& cred : credentials) {
        if (cred.first == username) {
            int key = 3; // Using the same fixed key
            string decryptedPassword = decrypt(cred.second, key);
            cout << "Password for " << username << ": " << decryptedPassword << endl;
            return;
        }
    }
    cout << "No password found for " << username << endl;
}

// Deletes credentials from the Password Manager
void PasswordManager::deleteCredentials() {
    string username;
    cout << "Enter username to delete credentials: ";
    cin >> username;

    // Use remove_if from <algorithm>
    auto it = remove_if(credentials.begin(), credentials.end(),
                        [&username](const pair<string, string>& cred) {
                            return cred.first == username;
                        });

    if (it != credentials.end()) {
        credentials.erase(it, credentials.end());
        cout << "Credentials for " << username << " deleted" << endl; // Fixed output stream
    } else {
        cout << "No credentials found for " << username << endl;
    }
}

// Saves credentials to a file - lec11
bool PasswordManager::saveToFile(const string& filename) {
    ofstream file(filename);
    if (!file.is_open()) {
        cout << "Error: Could not open file for saving." << endl; // Fixed output stream
        return false;
    }

    for (const auto& cred : credentials) {
        file << cred.first << endl;
        file << cred.second << endl;
    }

    file.close();
    cout << "Credentials saved to " << filename << endl;
    return true;
}
// lec11 if not credentials not saved
bool PasswordManager::loadFromFile(const string& filename) {
    ifstream file(filename);
    if (!file.is_open()) {
        cerr << "Error: Could not open file for loading." << endl; // Fixed output stream
        return false;
    }

    credentials.clear();

    string username;
    string password;
    while (getline(file, username) && getline(file, password)) { // Read username and encrypted password
        credentials.push_back(make_pair(username, password)); // Store encrypted password
    }

    file.close();
    cout << "Credentials loaded from " << filename << endl;
    return true;
}

int main() {
    PasswordManager manager; // Create an instance of PasswordManager class

    if (!manager.setMasterPassword()) {
        cout << "Failed to set master password. Exiting." << endl; // Fixed output stream
        return 1;
    }

    manager.loadFromFile();

    int choice;
    do {
        cout << "Password Manager Menu:" << endl;
        cout << "1. Add Credentials" << endl;
        cout << "2. Get Credentials" << endl;
        cout << "3. Delete Credentials" << endl;
        cout << "4. Save to File" << endl;
        cout << "5. Load from File" << endl;
        cout << "6. Exit" << endl;
        cout << "7. Add Credentials (Generated Password)" << endl;
        cout << "Enter your choice: " << endl; // Fixed output stream
        cin >> choice;

        switch (choice) {
            case 1:
                manager.addCredentials();
                break;
            case 2:
                manager.getCredentials();
                break;
            case 3:
                manager.deleteCredentials();
                break;
            case 4:
                manager.saveToFile();
                break;
            case 5:
                manager.loadFromFile();
                break;
            case 6:
                cout << "Exiting Password Manager." << endl; // Fixed output stream
                break;
            case 7:  // Handle the new option
                manager.addCredentialsWithGen();
                break;
            default:
                cout << "Invalid choice. Please try again." << endl; // Fixed output stream
        }
    } while (choice != 6);

    manager.saveToFile();

    return 0;
}


Writing password_manager.cpp


In [ ]:
!g++ password_manager.cpp -o password_manager
!./password_manager

Set your master password:(numbers only)9999
Error: Could not open file for loading.
Password Manager Menu:
1. Add Credentials
2. Get Credentials
3. Delete Credentials
4. Save to File
5. Load from File
6. Exit
7. Add Credentials (Generated Password)
Enter your choice: 
^C
